# Mandate Effect — Story Insights & Visualisations

**Angle:** The Oct 2023 staffing mandate lifted national quality by +7.4% — but fixed inputs, not outcomes.  
**Chapter:** 04_the_trend  
**Output:** Charts + confirmed numbers ready for dashboard callouts.

---
### Confirmed numbers (from 01_eda.ipynb)
| Finding | Number |
|---------|--------|
| National quality before mandate | 3.40 |
| National quality after mandate | 3.65 |
| Change | +0.25 pts (+7.4%) |
| Staffing sub-rating change | +0.51 pts (2.49 → 3.00) |
| Quality measures change | −0.015 pts |
| NT rank change | 7 → 1 (+0.748 pts) |
| VIC rank change | 1 → 5 (+0.312 pts) |
| SA3s still declining | 17 / 323 |

In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

CLEAN = '../../data/clean'

ratings = pd.read_csv(f'{CLEAN}/star_ratings_by_facility.csv', parse_dates=['snapshot_date'])

# Case-insensitive org_type mapping (Purpose column has mixed capitalisation across years)
ratings['org_type'] = ratings['Purpose'].str.strip().str.lower().map({
    'for profit': 'profit',
    'not for profit': 'not_for_profit',
    'government': 'government',
}).fillna('unknown')

MANDATE      = pd.Timestamp('2023-10-01')
MANDATE_TS   = MANDATE.timestamp() * 1000  # ms — required for plotly add_vline on datetime axis
latest_snap  = ratings['snapshot_date'].max()
first_snap   = ratings['snapshot_date'].min()

ratings['period'] = ratings['snapshot_date'].apply(
    lambda d: 'After mandate' if d >= MANDATE else 'Before mandate'
)

print(f'Rows: {len(ratings):,} | Snapshots: {ratings["snapshot_date"].nunique()}')
print(f'First: {first_snap.strftime("%B %Y")} | Latest: {latest_snap.strftime("%B %Y")}')

## 1. National Quality Step-Change

In [ ]:
# Confirm before/after numbers
before = ratings[ratings['period'] == 'Before mandate']['quality_score'].mean()
after  = ratings[ratings['period'] == 'After mandate']['quality_score'].mean()
print(f'Before mandate : {before:.3f}')
print(f'After mandate  : {after:.3f}')
print(f'Change         : {after - before:+.3f} pts  ({(after - before) / before * 100:+.1f}%)')

In [ ]:
# National avg quality per snapshot
qual_national = ratings.groupby('snapshot_date')['quality_score'].mean().reset_index()

fig = px.line(
    qual_national, x='snapshot_date', y='quality_score',
    title='National average quality score over time',
    labels={'quality_score': 'Avg quality score', 'snapshot_date': 'Quarter'},
    markers=True,
)
fig.add_vline(
    x=MANDATE_TS, line_dash='dash', line_color='red',
    annotation_text='Oct 2023 staffing mandate',
    annotation_position='top left',
)
fig.update_layout(yaxis_range=[3.0, 4.0], height=380)
fig.show()

In [ ]:
# Quality by state over time — for dashboard lead chart
qual_state = ratings.groupby(['snapshot_date', 'state'])['quality_score'].mean().reset_index()

fig = px.line(
    qual_state, x='snapshot_date', y='quality_score', color='state',
    title='Average quality score by state over time',
    labels={'quality_score': 'Avg quality score', 'snapshot_date': 'Quarter'},
    markers=True,
)
fig.add_vline(
    x=MANDATE_TS, line_dash='dash', line_color='red',
    annotation_text='Oct 2023 staffing mandate',
    annotation_position='top right',
)
fig.update_layout(yaxis_range=[2.5, 4.5], height=420)
fig.show()

> **Callout (st.success):** The Oct 2023 staffing mandate delivered a measurable result: national average quality rose from 3.40 to 3.65 — a +7.4% step-change visible across all states within two quarters.

## 2. Sub-Rating Decomposition — What Actually Moved?

In [ ]:
dims       = ['residents_exp', 'staffing', 'compliance', 'quality_measures']
dim_labels = ['Residents experience', 'Staffing', 'Compliance', 'Quality measures']

records = []
for d, label in zip(dims, dim_labels):
    b = ratings[ratings['period'] == 'Before mandate'][d].mean()
    a = ratings[ratings['period'] == 'After mandate'][d].mean()
    records.append({'dimension': label, 'before': round(b, 3), 'after': round(a, 3), 'change': round(a - b, 3)})

sub_df = pd.DataFrame(records).sort_values('change', ascending=False)
print(sub_df.to_string(index=False))

In [ ]:
fig = px.bar(
    sub_df.melt(id_vars='dimension', value_vars=['before', 'after'],
                var_name='period', value_name='score'),
    x='dimension', y='score', color='period', barmode='group',
    title='Sub-rating change before vs after Oct 2023 staffing mandate',
    labels={'score': 'Avg score', 'dimension': 'Sub-rating'},
    color_discrete_map={'before': '#aec7e8', 'after': '#1f77b4'},
    category_orders={'dimension': dim_labels},
)
fig.update_layout(yaxis_range=[2.0, 5.0], height=380)
fig.show()

> **Callout (st.info):** Staffing scores drove the gain: +0.51 pts (2.49 → 3.00), the largest jump of any dimension. But quality measures — which track resident health outcomes — moved just −0.015 pts. The mandate improved inputs. Outcomes have not yet followed.

## 3. State Rankings — Winners and Losers

In [ ]:
first_state = ratings[ratings['snapshot_date'] == first_snap].groupby('state')['quality_score'].mean()
last_state  = ratings[ratings['snapshot_date'] == latest_snap].groupby('state')['quality_score'].mean()

rank_df = pd.DataFrame({'first': first_state, 'last': last_state})
rank_df['change']      = rank_df['last'] - rank_df['first']
rank_df['rank_first']  = rank_df['first'].rank(ascending=False).astype(int)
rank_df['rank_last']   = rank_df['last'].rank(ascending=False).astype(int)
rank_df['rank_change'] = rank_df['rank_first'] - rank_df['rank_last']
rank_df = rank_df.reset_index().sort_values('change', ascending=False)

print(rank_df[['state', 'first', 'last', 'change', 'rank_first', 'rank_last', 'rank_change']].round(3).to_string(index=False))

In [ ]:
fig = px.bar(
    rank_df, x='state', y='change', color='change',
    color_continuous_scale='RdYlGn',
    title=f'Quality score change by state ({first_snap.strftime("%b %Y")} → {latest_snap.strftime("%b %Y")})',
    labels={'change': 'Change in avg quality score', 'state': 'State'},
    text='change',
)
fig.update_traces(texttemplate='%{text:+.3f}', textposition='outside')
fig.add_hline(y=0, line_color='black', line_width=1)
fig.update_layout(height=400)
fig.show()

In [ ]:
# Slope chart: first vs last per state
fig2 = go.Figure()
for _, row in rank_df.iterrows():
    color = '#2ca02c' if row['change'] > 0.5 else ('#d62728' if row['change'] < 0.35 else '#7f7f7f')
    fig2.add_trace(go.Scatter(
        x=[first_snap.strftime('%b %Y'), latest_snap.strftime('%b %Y')],
        y=[row['first'], row['last']],
        mode='lines+markers+text',
        name=row['state'],
        line=dict(color=color, width=2),
        text=[row['state'], row['state']],
        textposition=['middle left', 'middle right'],
    ))
fig2.update_layout(
    title='Quality slope chart — state first vs latest snapshot',
    yaxis_title='Avg quality score', showlegend=False,
    yaxis_range=[3.0, 4.5], height=420,
)
fig2.show()

> **Insight:** NT jumped from rank 7 to rank 1 (+0.748 pts) — a low base accelerated by the mandate. VIC fell from rank 1 to rank 5 despite improving +0.312 pts — other states caught up faster. No state declined in absolute terms.

## 4. SA3s Still Declining — Where the Mandate Hasn't Landed

In [ ]:
# Vectorized slope — avoids slow Python loop over each SA3
# Step 1: collapse to SA3 × snapshot mean (facility rows → 1 row per SA3 per quarter)
sa3_snap = (
    ratings.groupby(['sa3_code', 'sa3_name', 'snapshot_date'])['quality_score']
    .mean().reset_index()
    .sort_values(['sa3_code', 'snapshot_date'])
)

# Step 2: time index within each SA3
sa3_snap['t'] = sa3_snap.groupby('sa3_code').cumcount()

# Step 3: filter to SA3s with >= 4 snapshots
valid_sa3 = sa3_snap.groupby('sa3_code')['t'].max()
valid_sa3 = valid_sa3[valid_sa3 >= 3].index
sa3_snap  = sa3_snap[sa3_snap['sa3_code'].isin(valid_sa3)]

# Step 4: slope via numpy polyfit inside groupby apply
sa3_state = ratings[['sa3_code', 'state']].drop_duplicates('sa3_code')

slopes_df = (
    sa3_snap.groupby(['sa3_code', 'sa3_name'])
    .apply(lambda g: np.polyfit(g['t'], g['quality_score'], 1)[0])
    .reset_index(name='slope')
    .merge(sa3_state, on='sa3_code', how='left')
)

n_declining = (slopes_df['slope'] < 0).sum()
n_total     = len(slopes_df)
print(f'SA3s declining (slope < 0): {n_declining} / {n_total}')
print(f'SA3s improving (slope > 0): {(slopes_df["slope"] > 0).sum()} / {n_total}')
print()
print('Top 10 declining SA3s:')
print(slopes_df.nsmallest(10, 'slope')[['sa3_name', 'state', 'slope']].round(4).to_string(index=False))

In [ ]:
declining = slopes_df.nsmallest(15, 'slope')

fig = px.bar(
    declining, x='slope', y='sa3_name', color='state', orientation='h',
    title='SA3 regions with steepest quality decline — where the mandate has not landed',
    labels={'slope': 'Quality trend (pts per quarter)', 'sa3_name': 'SA3'},
)
fig.add_vline(x=0, line_color='red', line_dash='dash')
fig.update_layout(height=480)
fig.show()

> **Callout (st.warning):** 302 of 323 SA3 regions improved. But 17 are still declining — including Esperance WA at −0.07 pts/quarter. For these communities, the mandate has not landed.

---
## 5. Summary — Dashboard Callouts

```python
st.success(
    "The Oct 2023 staffing mandate delivered a measurable result: national average quality "
    "rose from 3.40 to 3.65 — a +7.4% step-change visible across all states within two quarters."
)

st.info(
    "Staffing scores drove the gain: +0.51 pts (2.49 → 3.00), the largest jump of any dimension. "
    "But quality measures — which track resident health outcomes — moved just −0.015 pts. "
    "The mandate improved inputs. Outcomes have not yet followed."
)

st.warning(
    "302 of 323 SA3 regions improved. But 17 are still declining — including Esperance WA "
    "at −0.07 pts/quarter. For these communities, the mandate has not landed."
)
```